# Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

## 1. Load Data

In [2]:
train=pd.read_csv('../data/raw/train.csv')
test=pd.read_csv('../data/raw/test.csv')

## 2. Drop unnecessary columns

Based on our EDA findings:

**Dropping due to high missing values (>60% threshold from proposal):**
- `neighbourhood_group_cleansed` - 100% 
- `host_neighbourhood` - 89% 
- `neighbourhood` - 71%  
- `neighborhood_overview` - 71%
- `host_about` - 61%

**Dropping non-predictive columns:**
- `Unnamed: 0.1`, `Unnamed: 0` - redundant index columns
- URL columns - not useful for prediction
- `scrape_id`, `last_scraped`, `source` - scraping metadata
- `description`, `name` - text column
- `license` - just license numbers
- `host_name`, `host_location` - useless 

In [3]:

drop_columns = [
    'Unnamed: 0.1', 'Unnamed: 0','neighbourhood_group_cleansed', 'host_neighbourhood', 'neighbourhood', 'neighborhood_overview', 'host_about',
    'listing_url', 'host_url', 'picture_url', 'host_thumbnail_url', 'host_picture_url','scrape_id', 'last_scraped', 'source',
    'description', 'name','license', 'host_name', 'host_location', 'host_verifications'
]

# drop train and test
train=train.drop(columns=drop_columns, errors='ignore')
test=test.drop(columns=drop_columns, errors='ignore')

print(f"train shape new:{train.shape}")
print(f"test shape new:{test.shape}")

train shape new:(24153, 37)
test shape new:(4750, 36)


## 3. Fix Data Types

In [4]:
train['price'] = train['price'].str.replace(',', '').astype(float)

In [5]:
# Convert percentage columns to numeric
train['host_response_rate'] = train['host_response_rate'].str.rstrip('%').astype(float)
train['host_acceptance_rate'] = train['host_acceptance_rate'].str.rstrip('%').astype(float)

test['host_response_rate'] = test['host_response_rate'].str.rstrip('%').astype(float)
test['host_acceptance_rate'] = test['host_acceptance_rate'].str.rstrip('%').astype(float)

print(train['host_response_rate'].head())

0      NaN
1      NaN
2      NaN
3    100.0
4    100.0
Name: host_response_rate, dtype: float64


In [6]:
# Convert boolean columns (t/f to 1/0)
bool_columns = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable']

for col in bool_columns:
    train[col] = train[col].map({'t': 1, 'f': 0})
    test[col] = test[col].map({'t': 1, 'f': 0})

print(train[bool_columns].head())

   host_is_superhost  host_has_profile_pic  host_identity_verified  \
0                0.0                   1.0                     1.0   
1                0.0                   NaN                     NaN   
2                0.0                   1.0                     1.0   
3                1.0                   1.0                     1.0   
4                0.0                   1.0                     1.0   

   instant_bookable  
0                 0  
1                 0  
2                 0  
3                 0  
4                 0  


## 4. Handle Missing Values

Strategy based on our proposal:
- **Numeric columns:** Fill with median
- **Categorical columns:** Fill with mode or 'Unknown'
- **Price (target):** Drop rows with missing price in train

In [7]:
# Drop rows with missing price (target variable) in train
print(f"before drop missing price:{train.shape}")
train = train.dropna(subset=['price'])
print(f"after drop missing price:{train.shape}")

before drop missing price:(24153, 37)
after drop missing price:(20804, 37)


In [9]:
# Fill numeric columns with median
numeric_cols = ['host_response_rate', 'host_acceptance_rate', 'host_listings_count',
                'host_total_listings_count', 'bathrooms', 'bedrooms', 'beds',
                'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
                'review_scores_checkin', 'review_scores_communication', 
                'review_scores_location', 'review_scores_value']

for col in numeric_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

print("missing value after filling")
print(train[numeric_cols].isnull().sum())

missing value after filling
host_response_rate             0
host_acceptance_rate           0
host_listings_count            0
host_total_listings_count      0
bathrooms                      0
bedrooms                       0
beds                           0
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_checkin          0
review_scores_communication    0
review_scores_location         0
review_scores_value            0
dtype: int64


In [10]:
# Fill boolean columns with mode
bool_columns = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']

for col in bool_columns:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    test[col] = test[col].fillna(mode_val)

print("missing value fter filling")
print(train[bool_columns].isnull().sum())

missing value fter filling
host_is_superhost         0
host_has_profile_pic      0
host_identity_verified    0
dtype: int64


In [12]:
# Fill host_response_time with mode
train['host_response_time']=train['host_response_time'].fillna('unknown')
test['host_response_time']=test['host_response_time'].fillna('unknown')

# Fill bathrooms_text with mode
mode_bathroom=train['bathrooms_text'].mode()[0]
train['bathrooms_text']=train['bathrooms_text'].fillna(mode_bathroom)
test['bathrooms_text']=test['bathrooms_text'].fillna(mode_bathroom)

# Fill host_since with mode 
mode_host_since = train['host_since'].mode()[0]
train['host_since'] = train['host_since'].fillna(mode_host_since)
test['host_since'] = test['host_since'].fillna(mode_host_since)

print("Remaining missing values:")
print(train.isnull().sum().sum())

Remaining missing values:
0


## 5. Feature Engineering

Creating new features based on our proposal:
- **host_experience_days:** Days since host joined Airbnb
- **price_per_person:** Price divided by accommodates
- **amenities_count:** Number of amenities
- **has_wifi, has_kitchen, has_ac:** Key amenity indicators

In [13]:
# Create host_experience_days (days since host joined)
from datetime import datetime

reference_date = datetime(2025, 7, 1)  # approximate scrape date

train['host_since'] = pd.to_datetime(train['host_since'])
test['host_since'] = pd.to_datetime(test['host_since'])

train['host_experience_days'] = (reference_date - train['host_since']).dt.days
test['host_experience_days'] = (reference_date - test['host_since']).dt.days

print("Host experience days sample:")
print(train['host_experience_days'].describe())

Host experience days sample:
count    20804.000000
mean      1899.919967
std       1285.815122
min          6.000000
25%        856.000000
50%       1573.000000
75%       2945.500000
max       5542.000000
Name: host_experience_days, dtype: float64


In [14]:
# Parse amenities and create features
train['amenities_count'] = train['amenities'].apply(lambda x: len(str(x).split(',')))
test['amenities_count'] = test['amenities'].apply(lambda x: len(str(x).split(',')))

# Key amenities indicators
train['has_wifi'] = train['amenities'].str.lower().str.contains('wifi').astype(int)
train['has_kitchen'] = train['amenities'].str.lower().str.contains('kitchen').astype(int)
train['has_ac'] = train['amenities'].str.lower().str.contains('air conditioning').astype(int)

test['has_wifi'] = test['amenities'].str.lower().str.contains('wifi').astype(int)
test['has_kitchen'] = test['amenities'].str.lower().str.contains('kitchen').astype(int)
test['has_ac'] = test['amenities'].str.lower().str.contains('air conditioning').astype(int)

print("Amenities features:")
print(f"Amenities count - mean: {train['amenities_count'].mean():.1f}")
print(f"Has WiFi: {train['has_wifi'].mean()*100:.1f}%")
print(f"Has Kitchen: {train['has_kitchen'].mean()*100:.1f}%")
print(f"Has AC: {train['has_ac'].mean()*100:.1f}%")

Amenities features:
Amenities count - mean: 29.5
Has WiFi: 96.1%
Has Kitchen: 84.3%
Has AC: 67.5%


In [15]:
# Create price_per_person (only for train)
train['price_per_person'] = train['price'] / train['accommodates']

print("Price per person sample:")
print(train['price_per_person'].describe())

Price per person sample:
count     20804.000000
mean       1318.438347
std        9408.736696
min          17.000000
25%         559.500000
50%         797.428571
75%        1166.666667
max      886200.000000
Name: price_per_person, dtype: float64


In [16]:
# Extract numeric value from bathrooms_text (e.g., "1.5 baths" -> 1.5)
def extract_bathroom(text):
    try:
        return float(str(text).split()[0])
    except:
        return 1.0

train['bathrooms_clean'] = train['bathrooms_text'].apply(extract_bathroom)
test['bathrooms_clean'] = test['bathrooms_text'].apply(extract_bathroom)

print("Bathrooms clean sample:")
print(train['bathrooms_clean'].value_counts().head(10))

Bathrooms clean sample:
bathrooms_clean
1.0    17104
2.0     2027
1.5      853
3.0      246
2.5      165
0.0      131
4.0       98
5.0       45
3.5       29
6.0       27
Name: count, dtype: int64


In [17]:
# Drop columns we no longer need
drop_after_fe = ['host_since', 'amenities', 'bathrooms_text', 'bathrooms']

train = train.drop(columns=drop_after_fe, errors='ignore')
test = test.drop(columns=drop_after_fe, errors='ignore')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (20804, 40)
Test shape: (4750, 38)


## 6. Encode Categorical Variables

Converting categorical columns to numeric using Label Encoding.

In [18]:
# Identify categorical columns
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {cat_cols}")

Categorical columns: ['host_response_time', 'neighbourhood_cleansed', 'property_type', 'room_type']


In [19]:
# Label encode categorical columns
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    # Fit on combined train + test to handle all categories
    combined = pd.concat([train[col], test[col]], axis=0)
    le.fit(combined)
    
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])
    label_encoders[col] = le

print("Categorical columns encoded:")
for col in cat_cols:
    print(f"{col}: {train[col].nunique()} unique values")

Categorical columns encoded:
host_response_time: 5 unique values
neighbourhood_cleansed: 39 unique values
property_type: 80 unique values
room_type: 4 unique values


## 7. Final Check and Save Data

In [20]:
# Final check - data types and missing values
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain dtypes:")
print(train.dtypes)
print("\nMissing values in train:", train.isnull().sum().sum())
print("Missing values in test:", test.isnull().sum().sum())

Train shape: (20804, 40)
Test shape: (4750, 38)

Train dtypes:
id                                                int64
host_id                                           int64
host_response_time                                int64
host_response_rate                              float64
host_acceptance_rate                            float64
host_is_superhost                               float64
host_listings_count                             float64
host_total_listings_count                       float64
host_has_profile_pic                            float64
host_identity_verified                          float64
neighbourhood_cleansed                            int64
latitude                                        float64
longitude                                       float64
property_type                                     int64
room_type                                         int64
accommodates                                      int64
bedrooms                                 

In [21]:
# Save processed data
train.to_csv('../data/processed/train_processed.csv', index=False)
test.to_csv('../data/processed/test_processed.csv', index=False)

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Train: (20804, 40)
Test: (4750, 38)


## Summary

| Step | Result |
|------|--------|
| Dropped columns | 58 → 37 |
| Dropped missing price rows | 24,153 → 20,804 |
| New features | 6 (host_experience_days, amenities_count, has_wifi, has_kitchen, has_ac, bathrooms_clean) |
| Final shape | Train: (20804, 40), Test: (4750, 38) |